# TP 1: Scraping de Votaciones de HCDN
---

|                 |                                                                                      |
| --              | --                                                                                   |
| **Universidad** | Universidad Nacional del Oeste                                                       |
| **Carrera**     | Esp. en Ciencia de Datos                                                             |
| **Materia**     | Aplicaciones de Ciencia de Datos (07102)                                             |
| **Profesor**    | Dr. Francisco Fernandez (franaf@gmail.com)                                           |
| **Alumnos**     | Grupo = {Lic. Leticia Sosa, Ing. Federico Czerniawski, Mg. Ing. Pablo Pandolfo}      |
| **Fecha**       | Octubre 2025                                                                         |
|                 |                                                                                      |

1. Abrir Chrome
2. Ir a https://votaciones.hcdn.gob.ar/
3. Hacer clic en "Ver más" hasta llegar a 2022
4. Extraer todas las votaciones (fecha, título, tipo, resultado, ID)
5. Guardar CSV principal con todas las votaciones
6. Descargar 10 CSVs individuales (para no tardar mucho)

In [6]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
import pandas as pd
import time

class Votaciones:

    def __init__(self, url):
        self.url = url
        self.driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()))

    def get_data(self, año_hasta, cant_descargas):
        json = {'data': []}
        self.driver.get(self.url)
        actas = self.driver.find_elements(By.XPATH, "//tbody[@id='container-actas']//tr")
        
        for i in range(len(actas)):
            try:
                actas = self.driver.find_elements(By.XPATH, "//tbody[@id='container-actas']//tr")      
            except StaleElementReferenceException:
                actas = self.driver.find_elements(By.XPATH, "//tbody[@id='container-actas']//tr")
            finally:
                acta = actas[i]
                id = acta.get_attribute('id')
                data = self.driver.find_elements(By.XPATH, f"//tbody[@id='container-actas']//tr[@id='{id}']//td") 
                fecha = data[0].text
                titulo = data[1].text
                tipo = data[2].text
                rtado = data[3].text
                
            if fecha.split('\n')[0].split('/')[2] == año_hasta:
                break
                
            json['data'].append({'id':id, 'fecha': fecha, 'titulo': titulo, 'tipo': tipo, 'rtado': rtado})

            # Descarga CSV
            if i < cant_descargas:
                time.sleep(1)
                self.driver.find_element(By.XPATH, f"//a[@href='/votacion/{id}']").click()
                time.sleep(3)
                self.driver.find_element(By.XPATH, "//a[@title='Descargar datos en CSV']").click()
                time.sleep(1)
                self.driver.find_element(By.XPATH, "//a[text()='Volver']").click()
                time.sleep(4)
            
            if i % 10 == 0:
                boton_ver_mas = self.driver.find_element(By.XPATH, "//*[text()='Ver más']")
                self.driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", boton_ver_mas)
                time.sleep(1)
                self.driver.execute_script("arguments[0].click();", boton_ver_mas)

        self.driver.close()
        return pd.DataFrame(json['data'])

vot = Votaciones('https://votaciones.hcdn.gob.ar/')
df = vot.get_data('2022', 10)
df.head()

,id,fecha,titulo,tipo,rtado
0,5793,09/10/2025\n02:44,O.D. 948 - ELIMINACIÓN DE DIVERSOS FIDEICOMISO...,Votación Nominal,NEGATIVO
1,5792,09/10/2025\n02:30,"O.D. 718 - BR. ESTANISLAO LÓPEZ ""PATRIARCA DEL...",Votación Nominal,AFIRMATIVO
2,5790,09/10/2025\n01:56,O.D. 950 - PROGRAMA NACIONAL DE LUCHA CONTRA L...,Votación Nominal,AFIRMATIVO
3,5789,09/10/2025\n01:48,O.D. 947 - EMERGENCIA Y FINANCIAMIENTO DEL SIS...,Votación Nominal,AFIRMATIVO
4,5791,09/10/2025\n01:22,EXPTE. 5408-D-2025. TEXTO PROPUESTO POR DIPUTA...,Votación Nominal,AFIRMATIVO


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 332 entries, 0 to 331
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      332 non-null    object
 1   fecha   332 non-null    object
 2   titulo  332 non-null    object
 3   tipo    332 non-null    object
 4   rtado   332 non-null    object
dtypes: object(5)
memory usage: 13.1+ KB
